<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Gemini API and VertexAI</h1>
<h1>Deployment Strategies</h1>
        <p>Bruno Gon&ccedil;alves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [ ]:
from collections import Counter
from pprint import pprint
import os
import time
import json
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import google.generativeai as genai
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from google.cloud import aiplatform

import watermark
%load_ext watermark
%matplotlib inline

We start by printing out the versions of the libraries we're using for future reference

In [ ]:
%watermark -n -v -m -g -iv

Load default figure style

In [ ]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Section 1: Advanced MLOps Practices with Vertex AI

Vertex AI provides a managed MLOps platform that covers the full ML lifecycle:
model training, evaluation, versioning, deployment, and monitoring. In this
section we walk through the key MLOps primitives available in the platform.

## 1.1 – Initialize Vertex AI

All Vertex AI SDK calls require the project ID and a region. We read the project
from an environment variable so that no credentials are hard-coded in the
notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Environment setup
# ---------------------------------------------------------------------------
# Before running this notebook make sure the following are set:
#   export GCP_PROJECT="your-gcp-project-id"
#   gcloud auth application-default login
# ---------------------------------------------------------------------------

GCP_PROJECT  = os.environ.get("GCP_PROJECT", "my-gcp-project")   # fallback for demo
GCP_LOCATION = "us-central1"

# Initialize the Vertex AI SDK.  All subsequent SDK calls will use these
# defaults unless overridden at the call site.
vertexai.init(project=GCP_PROJECT, location=GCP_LOCATION)

print(f"Vertex AI initialised")
print(f"  Project : {GCP_PROJECT}")
print(f"  Location: {GCP_LOCATION}")

## 1.2 – Vertex AI Pipelines (Kubeflow Pipelines)

Vertex AI Pipelines is a managed service that runs KFP (Kubeflow Pipelines)
graphs on GCP infrastructure. Pipelines are made of *components* – isolated,
containerised steps – that exchange typed artefacts.

The pattern:
1. Decorate a Python function with `@component` to turn it into a pipeline step.
2. Compose the steps inside a function decorated with `@pipeline`.
3. Compile the pipeline to a JSON spec and submit it to Vertex AI Pipelines.

In [ ]:
# ---------------------------------------------------------------------------
# NOTE: kfp (Kubeflow Pipelines SDK) must be installed:
#   pip install kfp google-cloud-pipeline-components
# The code below illustrates the authoring pattern; compilation requires the
# real KFP SDK.  We mock the decorators so the cell is always runnable.
# ---------------------------------------------------------------------------

try:
    from kfp import dsl
    from kfp.dsl import component, pipeline, Dataset, Model, Metrics, Output, Input
    KFP_AVAILABLE = True
except ImportError:
    KFP_AVAILABLE = False
    print("kfp not installed – showing code pattern only (no compilation).")

    # Minimal stubs so the cell is still syntactically valid
    class _Stub:
        def __call__(self, fn=None, **kw):
            return fn if fn else lambda f: f
        def __getattr__(self, name):
            return self

    class dsl:
        pipeline = _Stub()

    component = _Stub()
    pipeline  = _Stub()
    Dataset = Model = Metrics = Output = Input = object

print(f"KFP available: {KFP_AVAILABLE}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 1 – Data ingestion component
# ---------------------------------------------------------------------------
# Each @component function runs in its own container.  The base_image parameter
# controls which Docker image is used.  Dependencies listed in `packages_to_install`
# are pip-installed at runtime before the function body executes.
# ---------------------------------------------------------------------------

@component(
    base_image="python:3.11-slim",
    packages_to_install=["google-cloud-storage", "pandas"],
)
def ingest_data(
    gcs_uri: str,          # URI of the raw dataset (gs://bucket/path)
    dataset: Output[Dataset],   # typed output artefact – Vertex AI tracks provenance
) -> None:
    """Download raw data from GCS and write it as a pipeline artefact."""
    import pandas as pd
    from google.cloud import storage

    # Parse the GCS URI into bucket / blob
    parts      = gcs_uri.replace("gs://", "").split("/", 1)
    bucket_name, blob_name = parts[0], parts[1]

    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob   = bucket.blob(blob_name)
    blob.download_to_filename("/tmp/raw.csv")

    df = pd.read_csv("/tmp/raw.csv")
    print(f"Loaded {len(df)} rows from {gcs_uri}")

    # Write to the output artefact path so downstream components can read it
    df.to_csv(dataset.path, index=False)


# ---------------------------------------------------------------------------
# Step 2 – Model training component
# ---------------------------------------------------------------------------
@component(
    base_image="python:3.11-slim",
    packages_to_install=["scikit-learn", "pandas", "joblib"],
)
def train_model(
    dataset: Input[Dataset],   # consumes the artefact produced above
    model:   Output[Model],    # produces a Model artefact
    metrics: Output[Metrics],  # tracks evaluation scores
    n_estimators: int = 100,
) -> None:
    """Train a simple classifier and serialise it as a Model artefact."""
    import pandas as pd
    import joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    df = pd.read_csv(dataset.path)

    # Assume last column is the target
    X = df.iloc[:, :-1].values
    y = df.iloc[:,  -1].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    clf.fit(X_train, y_train)

    accuracy = accuracy_score(y_test, clf.predict(X_test))
    print(f"Validation accuracy: {accuracy:.4f}")

    # Log metrics – these appear in the Vertex AI Experiments UI
    metrics.log_metric("accuracy", accuracy)
    metrics.log_metric("n_estimators", n_estimators)

    # Serialise model next to the artefact path
    import os
    os.makedirs(model.path, exist_ok=True)
    joblib.dump(clf, os.path.join(model.path, "model.joblib"))


# ---------------------------------------------------------------------------
# Step 3 – Model registration component
# ---------------------------------------------------------------------------
@component(
    base_image="python:3.11-slim",
    packages_to_install=["google-cloud-aiplatform"],
)
def register_model(
    model:        Input[Model],
    project:      str,
    location:     str,
    display_name: str,
) -> str:
    """Upload the trained model to Vertex AI Model Registry."""
    from google.cloud import aiplatform

    aiplatform.init(project=project, location=location)

    registered = aiplatform.Model.upload(
        display_name=display_name,
        artifact_uri=model.uri,                          # GCS path of the artefact
        serving_container_image_uri=(
            "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"
        ),
    )

    print(f"Registered model resource name: {registered.resource_name}")
    return registered.resource_name


print("Pipeline components defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Compose the pipeline and compile to a JSON spec
# ---------------------------------------------------------------------------

@pipeline(
    name="gemini-mlops-demo-pipeline",
    description="End-to-end MLOps pipeline: ingest → train → register",
    pipeline_root=f"gs://{GCP_PROJECT}-pipelines/gemini-mlops",
)
def mlops_pipeline(
    gcs_data_uri:    str = f"gs://{GCP_PROJECT}-data/train.csv",
    n_estimators:    int = 100,
    model_name:      str = "gemini-demo-classifier",
):
    # Each step returns a task object; artefacts flow via .outputs
    ingest_task  = ingest_data(gcs_uri=gcs_data_uri)

    train_task   = train_model(
        dataset=ingest_task.outputs["dataset"],
        n_estimators=n_estimators,
    )

    register_task = register_model(
        model=train_task.outputs["model"],
        project=GCP_PROJECT,
        location=GCP_LOCATION,
        display_name=model_name,
    )


# Compile to a YAML/JSON spec that Vertex AI Pipelines can execute
if KFP_AVAILABLE:
    from kfp import compiler
    compiler.Compiler().compile(
        pipeline_func=mlops_pipeline,
        package_path="mlops_pipeline.yaml",
    )
    print("Pipeline compiled to mlops_pipeline.yaml")

    # Submit the pipeline run to Vertex AI
    # job = aiplatform.PipelineJob(
    #     display_name="gemini-mlops-demo",
    #     template_path="mlops_pipeline.yaml",
    # )
    # job.submit()
    print("Uncomment job.submit() to run on Vertex AI Pipelines.")
else:
    print("Pipeline definition shown above – install kfp to compile & submit.")

## 1.3 – Model Registry

The Vertex AI Model Registry is a central store for versioned model artefacts.
Every registered model has a resource name of the form
`projects/{project}/locations/{location}/models/{model_id}` and supports
multiple *versions*, labels, and evaluation metrics.

In [ ]:
# ---------------------------------------------------------------------------
# List models already in the registry
# ---------------------------------------------------------------------------
# We wrap the call in a try/except so that the notebook remains runnable
# without live GCP credentials (responses are mocked for the demo).
# ---------------------------------------------------------------------------

USE_MOCK = True   # Set to False when running against a real GCP project

if not USE_MOCK:
    # Real call – requires google-cloud-aiplatform >= 1.38
    models = aiplatform.Model.list(
        filter="display_name=gemini-demo-classifier",
        order_by="create_time desc",
    )
    for m in models:
        print(f"  {m.display_name}  version={m.version_id}  created={m.create_time}")
else:
    # ---- Mock response for demo purposes ------------------------------------
    mock_models = [
        {"display_name": "gemini-demo-classifier", "version_id": "3",
         "create_time": "2025-01-15T10:30:00Z", "state": "MODEL_STATE_READY"},
        {"display_name": "gemini-demo-classifier", "version_id": "2",
         "create_time": "2025-01-10T08:00:00Z", "state": "MODEL_STATE_READY"},
        {"display_name": "gemini-demo-classifier", "version_id": "1",
         "create_time": "2025-01-01T12:00:00Z", "state": "MODEL_STATE_READY"},
    ]
    print("Registered models (mock):")
    for m in mock_models:
        print(f"  {m['display_name']}  version={m['version_id']}  "
              f"created={m['create_time']}  state={m['state']}")

In [ ]:
# ---------------------------------------------------------------------------
# Register (upload) a new model version
# ---------------------------------------------------------------------------
# In a real workflow the artifact_uri would point to a GCS path containing
# the serialised model file(s).  The serving container image handles loading
# and serving the artefact over HTTP.
# ---------------------------------------------------------------------------

ARTIFACT_URI = f"gs://{GCP_PROJECT}-models/gemini-classifier/v4"

if not USE_MOCK:
    new_model = aiplatform.Model.upload(
        display_name="gemini-demo-classifier",
        artifact_uri=ARTIFACT_URI,
        serving_container_image_uri=(
            "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"
        ),
        serving_container_predict_route="/predict",
        serving_container_health_route="/health",
        labels={"team": "data4sci", "env": "production"},
    )
    print(f"Uploaded: {new_model.resource_name}")
else:
    print("[MOCK] Model uploaded to Vertex AI Model Registry")
    print(f"  Resource name : projects/{GCP_PROJECT}/locations/{GCP_LOCATION}/models/1234567890")
    print(f"  Artifact URI  : {ARTIFACT_URI}")
    print("  Version       : 4")

## 1.4 – Experiment Tracking

Vertex AI Experiments integrates with the Model Registry and Pipelines to give
you a unified view of training runs, hyperparameters, and evaluation metrics.

In [ ]:
# ---------------------------------------------------------------------------
# Initialise an experiment, log parameters and metrics
# ---------------------------------------------------------------------------

EXPERIMENT_NAME = "gemini-classifier-experiment"

if not USE_MOCK:
    # Create (or resume) the experiment
    aiplatform.init(
        project=GCP_PROJECT,
        location=GCP_LOCATION,
        experiment=EXPERIMENT_NAME,
    )

    # Each run corresponds to one training trial
    with aiplatform.start_run(run=f"run-{datetime.now().strftime('%Y%m%d-%H%M%S')}"):
        # Log hyperparameters
        aiplatform.log_params({
            "n_estimators": 200,
            "max_depth":    10,
            "learning_rate": 0.05,
        })

        # Log evaluation metrics
        aiplatform.log_metrics({
            "accuracy":  0.9340,
            "precision": 0.9210,
            "recall":    0.9455,
            "f1":        0.9331,
        })

        print("Metrics and params logged to Vertex AI Experiments.")
else:
    # ---- Mock experiment output --------------------------------------------
    experiment_record = {
        "experiment": EXPERIMENT_NAME,
        "run":        f"run-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
        "params":  {"n_estimators": 200, "max_depth": 10, "learning_rate": 0.05},
        "metrics": {"accuracy": 0.9340, "precision": 0.9210,
                    "recall": 0.9455, "f1": 0.9331},
    }
    print("[MOCK] Experiment record:")
    pprint(experiment_record)

In [ ]:
# ---------------------------------------------------------------------------
# Visualise experiment results across multiple runs
# ---------------------------------------------------------------------------
# In a real environment you would call:
#   df = aiplatform.get_experiment_df(experiment=EXPERIMENT_NAME)
# Here we build mock data to illustrate the analysis pattern.
# ---------------------------------------------------------------------------

mock_experiment_df = pd.DataFrame({
    "run_name":          [f"run-{i:02d}" for i in range(1, 9)],
    "param.n_estimators": [50, 100, 150, 200, 250, 300, 350, 400],
    "metric.accuracy":    [0.881, 0.907, 0.921, 0.934, 0.938, 0.940, 0.941, 0.941],
    "metric.f1":          [0.872, 0.899, 0.914, 0.933, 0.937, 0.939, 0.940, 0.940],
})

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mock_experiment_df["param.n_estimators"],
        mock_experiment_df["metric.accuracy"],
        marker="o", label="Accuracy")
ax.plot(mock_experiment_df["param.n_estimators"],
        mock_experiment_df["metric.f1"],
        marker="s", linestyle="--", label="F1 Score")
ax.set_xlabel("Number of Estimators")
ax.set_ylabel("Score")
ax.set_title("Experiment Tracking: Performance vs Complexity")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Section 2: Model Monitoring and Explainability on Vertex AI

Once a model is serving live traffic, data distributions shift, model performance
degrades, and predictions may become less reliable. Vertex AI Model Monitoring
automates the detection of *training-serving skew* (difference between training
distribution and live traffic) and *prediction drift* (change in serving
distribution over time).

## 2.1 – Setting Up Model Monitoring

In [ ]:
# ---------------------------------------------------------------------------
# Configure a ModelDeploymentMonitoringJob
# ---------------------------------------------------------------------------
# This is the GCP object that periodically analyses logged prediction
# requests and compares them against a *baseline* (usually the training
# dataset statistics stored in BigQuery or GCS).
# ---------------------------------------------------------------------------

from google.cloud.aiplatform_v1.types import (
    ModelDeploymentMonitoringJob,
    ModelDeploymentMonitoringObjectiveConfig,
    ModelDeploymentMonitoringScheduleConfig,
    SamplingStrategy,
    ThresholdConfig,
)

ENDPOINT_ID = "1234567890"   # replace with your deployed endpoint ID

# Sampling strategy – log 10 % of prediction requests to BigQuery
sampling_strategy = {
    "random_sample_config": {"sample_rate": 0.10}
}

# Check every 1 hour
schedule_config = {
    "monitor_interval": {"seconds": 3600}
}

# Skew & drift thresholds per feature
objective_config = {
    "training_dataset": {
        "gcs_source": {"uris": [f"gs://{GCP_PROJECT}-data/train.csv"]},
        "target_field": "label",
    },
    "training_prediction_skew_detection_config": {
        "skew_thresholds": {
            "feature_1": {"value": 0.10},   # Jensen-Shannon divergence threshold
            "feature_2": {"value": 0.15},
        }
    },
    "prediction_drift_detection_config": {
        "drift_thresholds": {
            "feature_1": {"value": 0.10},
            "feature_2": {"value": 0.15},
        }
    },
}

monitoring_job_spec = {
    "display_name":       "gemini-classifier-monitoring",
    "endpoint":           f"projects/{GCP_PROJECT}/locations/{GCP_LOCATION}/endpoints/{ENDPOINT_ID}",
    "model_deployment_monitoring_objective_configs": [objective_config],
    "model_deployment_monitoring_schedule_config":    schedule_config,
    "logging_sampling_strategy":                      sampling_strategy,
    "enable_monitoring_pipeline_logs":                True,
    "log_ttl":                                        {"seconds": 60 * 60 * 24 * 30},  # 30 days
}

print("Monitoring job specification (would be submitted via aiplatform client):")
pprint(monitoring_job_spec)

## 2.2 – Feature Importance and Explainability

Vertex AI Explainable AI supports three explanation methods:
- **Sampled Shapley** – approximates SHAP values using random feature permutations.
- **Integrated Gradients** – suited for neural networks with continuous inputs or images.
- **XRAI** – image segmentation-based attribution.

For tabular models we typically use Sampled Shapley.

In [ ]:
# ---------------------------------------------------------------------------
# Configure explanations when uploading a model
# ---------------------------------------------------------------------------

explanation_spec = {
    "parameters": {
        "sampled_shapley_attribution": {
            "path_count": 10   # number of random permutation paths
        }
    },
    "metadata": {
        # Map input features to their names
        "inputs": {
            "feature_1": {"input_tensor_name": "feature_1"},
            "feature_2": {"input_tensor_name": "feature_2"},
            "feature_3": {"input_tensor_name": "feature_3"},
        },
        "outputs": {
            "scores": {"output_tensor_name": "scores"}
        }
    }
}

if not USE_MOCK:
    explained_model = aiplatform.Model.upload(
        display_name="gemini-demo-with-xai",
        artifact_uri=ARTIFACT_URI,
        serving_container_image_uri=(
            "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"
        ),
        explanation_parameters=explanation_spec["parameters"],
        explanation_metadata=explanation_spec["metadata"],
    )
    print(f"Model with XAI uploaded: {explained_model.resource_name}")
else:
    print("[MOCK] Model with XAI configuration would be uploaded with spec:")
    pprint(explanation_spec)

In [ ]:
# ---------------------------------------------------------------------------
# Visualise feature attributions returned by an explain() call
# ---------------------------------------------------------------------------
# In production: endpoint.explain(instances=[...]).explanations[0].attributions
# Here we simulate a realistic response.
# ---------------------------------------------------------------------------

features = [
    "prompt_length",
    "user_retention_days",
    "session_count",
    "avg_tokens_per_call",
    "error_rate",
    "latency_p99_ms",
    "daily_api_calls",
]
# Positive attributions push the prediction toward class 1;
# negative attributions push toward class 0
attributions = np.array([0.32, 0.27, 0.18, -0.14, -0.22, 0.09, 0.06])

# Sort by absolute value for readability
order = np.argsort(np.abs(attributions))[::-1]
features_sorted     = [features[i] for i in order]
attributions_sorted = attributions[order]

colors = ["steelblue" if v >= 0 else "tomato" for v in attributions_sorted]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(features_sorted, attributions_sorted, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("SHAP Attribution (Sampled Shapley)")
ax.set_title("Feature Importance for Single Prediction")
ax.invert_yaxis()   # largest bar at top
plt.tight_layout()
plt.show()

## 2.3 – Logging, Audit Trails, and Alerts

Vertex AI writes structured logs to Cloud Logging.  You can create log-based
metrics and wire them to Cloud Monitoring alerting policies.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated monitoring dashboard – drift over time
# ---------------------------------------------------------------------------
# Each point represents one monitoring window (e.g., 1-hour interval).
# The Jensen-Shannon divergence measures how much the serving distribution
# has shifted relative to the training baseline.
# ---------------------------------------------------------------------------

np.random.seed(42)
n_windows = 30

timestamps = pd.date_range(start="2025-01-01", periods=n_windows, freq="D")

# Simulate gradual drift with occasional spikes (e.g., traffic change)
baseline_drift  = np.linspace(0.02, 0.12, n_windows)
noise           = np.random.normal(0, 0.01, n_windows)
spikes          = np.zeros(n_windows)
spikes[[10, 22]] = [0.08, 0.12]   # two anomalous events

feature_1_drift = np.clip(baseline_drift + noise + spikes,         0, 1)
feature_2_drift = np.clip(baseline_drift * 0.7 + noise * 0.5,     0, 1)

DRIFT_THRESHOLD = 0.10

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(timestamps, feature_1_drift, marker="o", markersize=4, label="feature_1 JS-divergence")
ax.plot(timestamps, feature_2_drift, marker="s", markersize=4, linestyle="--", label="feature_2 JS-divergence")
ax.axhline(DRIFT_THRESHOLD, color="red", linestyle=":", linewidth=1.5, label=f"Alert threshold ({DRIFT_THRESHOLD})")
ax.fill_between(timestamps, DRIFT_THRESHOLD, 1,
                where=(feature_1_drift > DRIFT_THRESHOLD),
                alpha=0.15, color="red", label="Alert zone")
ax.set_xlabel("Date")
ax.set_ylabel("Jensen-Shannon Divergence")
ax.set_title("Vertex AI Model Monitoring – Feature Drift Over Time")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Alert policy skeleton (Cloud Monitoring API)
# ---------------------------------------------------------------------------
# In production you would apply this via the gcloud CLI or Terraform:
#   gcloud alpha monitoring policies create --policy-from-file=alert.json
# ---------------------------------------------------------------------------

alert_policy = {
    "displayName": "Gemini Classifier – Feature Drift Alert",
    "conditions": [
        {
            "displayName": "JS Divergence > 0.10 for 2 consecutive windows",
            "conditionThreshold": {
                "filter": (
                    'resource.type="aiplatform.googleapis.com/Endpoint" '
                    'metric.type="aiplatform.googleapis.com/prediction/online/'
                    'model_monitoring/feature_skew_metrics"'
                ),
                "comparison":          "COMPARISON_GT",
                "thresholdValue":       0.10,
                "duration":            {"seconds": 7200},   # 2 x 1-hour windows
                "aggregations": [
                    {"alignmentPeriod": {"seconds": 3600},
                     "perSeriesAligner": "ALIGN_MAX"}
                ],
            }
        }
    ],
    "notificationChannels": ["projects/my-project/notificationChannels/abc123"],
    "alertStrategy": {"autoClose": {"seconds": 86400}},  # auto-close after 24 h
}

print("Alert policy JSON:")
print(json.dumps(alert_policy, indent=2))

# Section 3: Deploying Gemini-powered Applications

This section covers the practical patterns for wrapping Gemini in production
services: Vertex AI managed endpoints, FastAPI micro-services, Cloud Run
containers, and client-side best practices such as caching and rate limiting.

## 3.1 – Deploy a Gemini Model Endpoint on Vertex AI

In [ ]:
# ---------------------------------------------------------------------------
# For Gemini models the SDK handles the endpoint internally – you call the
# model directly.  The snippet below shows how you would deploy a *custom*
# fine-tuned Gemini checkpoint via Vertex AI Endpoints.
# ---------------------------------------------------------------------------

def create_gemini_endpoint(
    project: str,
    location: str,
    model_resource_name: str,
    display_name: str = "gemini-production-endpoint",
    machine_type: str = "n1-standard-4",
    min_replica_count: int = 1,
    max_replica_count: int = 10,
    use_mock: bool = True,
):
    """Create a Vertex AI endpoint and deploy a model to it."""
    if use_mock:
        mock_endpoint_id = "9876543210"
        print(f"[MOCK] Endpoint created: {display_name}")
        print(f"  Endpoint ID   : {mock_endpoint_id}")
        print(f"  Model         : {model_resource_name}")
        print(f"  Machine type  : {machine_type}")
        print(f"  Replicas      : {min_replica_count} – {max_replica_count}")
        return {"endpoint_id": mock_endpoint_id, "display_name": display_name}

    aiplatform.init(project=project, location=location)

    # 1. Create the endpoint resource
    endpoint = aiplatform.Endpoint.create(
        display_name=display_name,
        labels={"env": "production", "team": "data4sci"},
    )

    # 2. Fetch the registered model
    model = aiplatform.Model(model_resource_name)

    # 3. Deploy the model to the endpoint with auto-scaling
    model.deploy(
        endpoint=endpoint,
        deployed_model_display_name=display_name,
        machine_type=machine_type,
        min_replica_count=min_replica_count,
        max_replica_count=max_replica_count,
        traffic_split={"0": 100},  # 100 % to this version
        accelerator_type=None,     # use GPU/TPU for large models
    )

    return endpoint


endpoint_info = create_gemini_endpoint(
    project=GCP_PROJECT,
    location=GCP_LOCATION,
    model_resource_name=f"projects/{GCP_PROJECT}/locations/{GCP_LOCATION}/models/1234567890",
    use_mock=USE_MOCK,
)

## 3.2 – FastAPI Integration

FastAPI is the recommended framework for wrapping Gemini in a production REST
API.  The code below shows a complete, runnable service (`app.py`).  When
deployed to Cloud Run it handles authentication, request validation, and
structured JSON responses.

In [ ]:
# ---------------------------------------------------------------------------
# app.py – FastAPI wrapper for Gemini
# ---------------------------------------------------------------------------
# To run locally:
#   pip install fastapi uvicorn[standard] google-generativeai
#   uvicorn app:app --reload --port 8080
# ---------------------------------------------------------------------------

FASTAPI_APP_CODE = '''
from __future__ import annotations

import os
import time
import logging
from typing import Optional

from fastapi import FastAPI, HTTPException, Depends, Security
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel, Field
import google.generativeai as genai

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Gemini client – initialised once at startup
# ---------------------------------------------------------------------------
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")

# ---------------------------------------------------------------------------
# FastAPI app
# ---------------------------------------------------------------------------
app = FastAPI(
    title="Gemini API Gateway",
    version="1.0.0",
    description="Production REST API wrapping Gemini on Vertex AI",
)

security = HTTPBearer()


# ---------------------------------------------------------------------------
# Request / response schemas
# ---------------------------------------------------------------------------
class GenerateRequest(BaseModel):
    prompt:      str                  = Field(...,  min_length=1, max_length=32_000)
    max_tokens:  int                  = Field(1024, ge=1, le=8192)
    temperature: float                = Field(0.7,  ge=0.0, le=2.0)
    system_prompt: Optional[str]      = None


class GenerateResponse(BaseModel):
    text:          str
    input_tokens:  int
    output_tokens: int
    latency_ms:    float
    model:         str


# ---------------------------------------------------------------------------
# Token-based auth (replace with IAM / OAuth2 in production)
# ---------------------------------------------------------------------------
VALID_TOKENS = {os.environ.get("API_TOKEN", "dev-token")}

def verify_token(credentials: HTTPAuthorizationCredentials = Security(security)):
    if credentials.credentials not in VALID_TOKENS:
        raise HTTPException(status_code=401, detail="Invalid or missing token")
    return credentials.credentials


# ---------------------------------------------------------------------------
# Endpoints
# ---------------------------------------------------------------------------
@app.get("/health")
def health():
    return {"status": "ok", "model": "gemini-1.5-pro"}


@app.post("/v1/generate", response_model=GenerateResponse)
def generate(
    request: GenerateRequest,
    _token: str = Depends(verify_token),
):
    """Generate text using Gemini."""
    t0 = time.perf_counter()

    try:
        generation_config = genai.types.GenerationConfig(
            max_output_tokens=request.max_tokens,
            temperature=request.temperature,
        )

        full_prompt = request.prompt
        if request.system_prompt:
            full_prompt = f"{request.system_prompt}\\n\\n{request.prompt}"

        response = model.generate_content(
            full_prompt,
            generation_config=generation_config,
        )

        latency_ms = (time.perf_counter() - t0) * 1000
        logger.info("generate latency=%.1f ms", latency_ms)

        return GenerateResponse(
            text=response.text,
            input_tokens=response.usage_metadata.prompt_token_count,
            output_tokens=response.usage_metadata.candidates_token_count,
            latency_ms=round(latency_ms, 1),
            model="gemini-1.5-pro",
        )

    except Exception as e:
        logger.exception("Generation failed")
        raise HTTPException(status_code=500, detail=str(e))
'''

print(FASTAPI_APP_CODE)

## 3.3 – Cloud Run Deployment Pattern

Cloud Run is the recommended compute platform for stateless Gemini API services.
It auto-scales to zero (cost-efficient) and to thousands of instances (elastic).

In [ ]:
# ---------------------------------------------------------------------------
# Dockerfile for the FastAPI service
# ---------------------------------------------------------------------------

DOCKERFILE = '''
# ---- Build stage -----------------------------------------------------------
FROM python:3.11-slim AS build

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ---- Runtime stage ---------------------------------------------------------
FROM python:3.11-slim

# Non-root user for security
RUN useradd --create-home appuser
USER appuser
WORKDIR /app

COPY --from=build /usr/local/lib/python3.11/site-packages /usr/local/lib/python3.11/site-packages
COPY --from=build /usr/local/bin /usr/local/bin
COPY app.py .

# Cloud Run expects the service to listen on PORT (default 8080)
ENV PORT=8080
EXPOSE 8080

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080", \
     "--workers", "4", "--timeout-keep-alive", "75"]
'''

CLOUD_RUN_DEPLOY_CMD = f"""
# Build and push the container image to Artifact Registry
gcloud builds submit --tag us-central1-docker.pkg.dev/{GCP_PROJECT}/gemini-api/service:latest

# Deploy to Cloud Run with auto-scaling and concurrency settings
gcloud run deploy gemini-api-service \\
  --image us-central1-docker.pkg.dev/{GCP_PROJECT}/gemini-api/service:latest \\
  --platform managed \\
  --region us-central1 \\
  --memory 2Gi \\
  --cpu 2 \\
  --concurrency 80 \\
  --min-instances 1 \\
  --max-instances 100 \\
  --set-env-vars GOOGLE_API_KEY=$$GOOGLE_API_KEY \\
  --no-allow-unauthenticated     # require IAM authentication
"""

print("Dockerfile:")
print(DOCKERFILE)
print("\nCloud Run deployment commands:")
print(CLOUD_RUN_DEPLOY_CMD)

## 3.4 – Rate Limiting and Caching

In [ ]:
# ---------------------------------------------------------------------------
# Simple in-process rate limiter (token-bucket algorithm)
# ---------------------------------------------------------------------------
# For production use a distributed rate limiter backed by Redis or
# Memorystore so the limit is enforced across all Cloud Run instances.
# ---------------------------------------------------------------------------

import threading

class TokenBucketRateLimiter:
    """Thread-safe token-bucket rate limiter."""

    def __init__(self, rate: float, capacity: float):
        """
        Parameters
        ----------
        rate     : tokens added per second
        capacity : maximum tokens in the bucket (burst size)
        """
        self.rate     = rate
        self.capacity = capacity
        self.tokens   = capacity
        self.last_refill = time.monotonic()
        self._lock    = threading.Lock()

    def _refill(self):
        now    = time.monotonic()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_refill = now

    def acquire(self, tokens: float = 1.0) -> bool:
        """Attempt to consume `tokens` from the bucket. Returns True on success."""
        with self._lock:
            self._refill()
            if self.tokens >= tokens:
                self.tokens -= tokens
                return True
            return False


# 10 requests per second, burst of 20
limiter = TokenBucketRateLimiter(rate=10, capacity=20)

# Simulate 15 rapid requests
allowed = rejected = 0
for i in range(15):
    if limiter.acquire():
        allowed  += 1
    else:
        rejected += 1

print(f"Allowed: {allowed}  Rejected: {rejected}  "
      f"(bucket capacity=20, burst handled correctly)")

In [ ]:
# ---------------------------------------------------------------------------
# Response cache with TTL (Time-to-Live)
# ---------------------------------------------------------------------------
# Identical prompts within the TTL window return cached responses instantly,
# eliminating redundant API calls and reducing cost.
#
# For production: replace the dict with Redis / Memorystore so the cache is
# shared across instances and survives restarts.
# ---------------------------------------------------------------------------

import hashlib
from dataclasses import dataclass, field

@dataclass
class CacheEntry:
    response:   str
    created_at: float = field(default_factory=time.monotonic)


class ResponseCache:
    """LRU-style response cache with per-entry TTL."""

    def __init__(self, ttl_seconds: int = 300, max_size: int = 1000):
        self.ttl       = ttl_seconds
        self.max_size  = max_size
        self._store: dict[str, CacheEntry] = {}
        self._lock = threading.Lock()

    @staticmethod
    def _key(prompt: str, model: str, temperature: float) -> str:
        """Deterministic cache key – temperature must be 0 for reproducibility."""
        raw = f"{model}|{temperature}|{prompt}"
        return hashlib.sha256(raw.encode()).hexdigest()

    def get(self, prompt: str, model: str, temperature: float = 0.0):
        key = self._key(prompt, model, temperature)
        with self._lock:
            entry = self._store.get(key)
            if entry is None:
                return None
            if time.monotonic() - entry.created_at > self.ttl:
                del self._store[key]
                return None
            return entry.response

    def set(self, prompt: str, model: str, temperature: float, response: str):
        key = self._key(prompt, model, temperature)
        with self._lock:
            if len(self._store) >= self.max_size:
                # Evict the oldest entry
                oldest = min(self._store, key=lambda k: self._store[k].created_at)
                del self._store[oldest]
            self._store[key] = CacheEntry(response=response)

    def stats(self) -> dict:
        with self._lock:
            return {"size": len(self._store), "max_size": self.max_size, "ttl": self.ttl}


# Demo – show cache hit/miss behaviour
cache = ResponseCache(ttl_seconds=60)

prompt = "Explain transformer attention in one paragraph."

# First call – cache miss
cached = cache.get(prompt, model="gemini-1.5-pro", temperature=0.0)
if cached is None:
    print("Cache MISS – calling API ...")
    # (In real code: response = model.generate_content(prompt).text)
    response_text = "[Simulated Gemini response about transformer attention]"
    cache.set(prompt, model="gemini-1.5-pro", temperature=0.0, response=response_text)
else:
    print("Cache HIT")

# Second call – cache hit
cached = cache.get(prompt, model="gemini-1.5-pro", temperature=0.0)
if cached:
    print(f"Cache HIT – returned instantly: {cached[:60]}...")

print(f"Cache stats: {cache.stats()}")

## 3.5 – Cost Estimation and Tracking

In [ ]:
# ---------------------------------------------------------------------------
# Token counting and cost estimation
# ---------------------------------------------------------------------------
# Gemini pricing (as of early 2025, check ai.google.dev/pricing for latest):
#   gemini-1.5-pro  input:  $3.50 / 1M tokens  (>128K context: $7.00)
#   gemini-1.5-pro  output: $10.50 / 1M tokens  (>128K context: $21.00)
#   gemini-1.5-flash input:  $0.075 / 1M tokens
#   gemini-1.5-flash output: $0.30  / 1M tokens
# ---------------------------------------------------------------------------

PRICING = {
    "gemini-1.5-pro": {
        "input_per_million":  3.50,
        "output_per_million": 10.50,
        "long_context_threshold": 128_000,   # tokens – price doubles above this
        "input_long_per_million":  7.00,
        "output_long_per_million": 21.00,
    },
    "gemini-1.5-flash": {
        "input_per_million":  0.075,
        "output_per_million": 0.30,
        "long_context_threshold": 128_000,
        "input_long_per_million":  0.15,
        "output_long_per_million": 0.60,
    },
}


def estimate_cost(
    model_name:    str,
    input_tokens:  int,
    output_tokens: int,
) -> dict:
    """Return cost breakdown in USD for a single API call."""
    p = PRICING.get(model_name, PRICING["gemini-1.5-pro"])

    if input_tokens > p["long_context_threshold"]:
        in_rate  = p["input_long_per_million"]
        out_rate = p["output_long_per_million"]
    else:
        in_rate  = p["input_per_million"]
        out_rate = p["output_per_million"]

    input_cost  = (input_tokens  / 1_000_000) * in_rate
    output_cost = (output_tokens / 1_000_000) * out_rate

    return {
        "model":         model_name,
        "input_tokens":  input_tokens,
        "output_tokens": output_tokens,
        "input_cost_usd":  round(input_cost,  6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd":  round(input_cost + output_cost, 6),
    }


# Example: 1000 requests/day with 500 input tokens and 300 output tokens
single_call   = estimate_cost("gemini-1.5-pro",   input_tokens=500, output_tokens=300)
single_flash  = estimate_cost("gemini-1.5-flash",  input_tokens=500, output_tokens=300)
daily_calls   = 1_000

print("Cost per API call:")
print(f"  gemini-1.5-pro  : ${single_call['total_cost_usd']:.6f}")
print(f"  gemini-1.5-flash: ${single_flash['total_cost_usd']:.6f}")
print()
print(f"Estimated daily cost at {daily_calls:,} calls/day:")
print(f"  gemini-1.5-pro  : ${single_call['total_cost_usd'] * daily_calls:.4f}")
print(f"  gemini-1.5-flash: ${single_flash['total_cost_usd'] * daily_calls:.4f}")
print()
print("Savings from using Flash: "
      f"{(1 - single_flash['total_cost_usd'] / single_call['total_cost_usd']) * 100:.1f}%")

# Section 4: Scaling and Optimization for Production

Moving from a prototype to a production system requires attention to throughput,
latency, and cost.  This section covers the key patterns: request batching,
async calls, response streaming, prompt caching, and token optimisation.

## 4.1 – Batching Requests for Throughput

In [ ]:
# ---------------------------------------------------------------------------
# Dynamic batching – accumulate requests and flush when full or timeout
# ---------------------------------------------------------------------------
# Vertex AI Batch Prediction processes large datasets offline without
# keeping connections open.  For online serving, "micro-batching" is a
# common pattern: buffer incoming requests for a short window and send
# them as one multi-prompt call (where the model API supports it).
# ---------------------------------------------------------------------------

import asyncio
from typing import List, Dict, Any


class MicroBatcher:
    """Collects requests and flushes them as a batch at a regular interval."""

    def __init__(self, max_batch_size: int = 8, flush_interval_ms: float = 50):
        self.max_batch_size    = max_batch_size
        self.flush_interval    = flush_interval_ms / 1000.0
        self._queue: List[Dict[str, Any]] = []
        self._futures: List[asyncio.Future] = []
        self._lock  = asyncio.Lock()

    async def add(self, prompt: str) -> str:
        """Add a prompt to the batch; await the response."""
        loop   = asyncio.get_event_loop()
        future = loop.create_future()
        async with self._lock:
            self._queue.append({"prompt": prompt})
            self._futures.append(future)
            if len(self._queue) >= self.max_batch_size:
                await self._flush()
        return await future

    async def _flush(self):
        """Process the current batch and resolve all pending futures."""
        if not self._queue:
            return
        batch   = self._queue[:]
        futures = self._futures[:]
        self._queue.clear()
        self._futures.clear()

        # In production: send all prompts to the model API
        # responses = await asyncio.gather(*[call_model(item["prompt"]) for item in batch])
        responses = [f"[Response to: {item['prompt'][:40]}]"
                     for item in batch]   # mock

        for future, response in zip(futures, responses):
            if not future.done():
                future.set_result(response)

    async def start_periodic_flush(self):
        """Background task that flushes on interval."""
        while True:
            await asyncio.sleep(self.flush_interval)
            async with self._lock:
                await self._flush()


print("MicroBatcher class defined – see Section 4.2 for async usage example.")

## 4.2 – Async API Calls with asyncio

In [ ]:
# ---------------------------------------------------------------------------
# Async concurrent requests with bounded semaphore (connection pooling)
# ---------------------------------------------------------------------------
# We use a semaphore to cap the number of in-flight requests and avoid
# hitting rate limits.  asyncio.gather executes all coroutines concurrently
# within the semaphore budget.
# ---------------------------------------------------------------------------

import asyncio
import random

MAX_CONCURRENT = 5   # honour Gemini QPM quota per minute

async def call_gemini_mock(prompt: str, request_id: int) -> dict:
    """
    Simulates an async Gemini API call with realistic latency jitter.
    In production replace with:
        model = genai.GenerativeModel('gemini-1.5-pro')
        response = await model.generate_content_async(prompt)
    """
    latency = random.uniform(0.3, 1.2)   # seconds
    await asyncio.sleep(latency)
    return {
        "id":      request_id,
        "prompt":  prompt[:40],
        "latency": latency,
        "tokens":  random.randint(50, 500),
    }


async def run_concurrent_batch(
    prompts: List[str],
    max_concurrent: int = MAX_CONCURRENT,
) -> List[dict]:
    """Execute all prompts concurrently, respecting the concurrency limit."""
    semaphore = asyncio.Semaphore(max_concurrent)

    async def bounded_call(prompt: str, idx: int):
        async with semaphore:
            return await call_gemini_mock(prompt, idx)

    tasks = [bounded_call(p, i) for i, p in enumerate(prompts)]
    return await asyncio.gather(*tasks)


# Run the async batch
sample_prompts = [
    "Summarise the history of neural networks.",
    "Explain the transformer architecture.",
    "What is Vertex AI?",
    "How does RLHF work?",
    "Describe few-shot prompting.",
    "What are embedding models used for?",
    "Explain the difference between LLM fine-tuning and RAG.",
    "What is the role of temperature in generation?",
]

t_start   = time.perf_counter()
results   = asyncio.run(run_concurrent_batch(sample_prompts, max_concurrent=MAX_CONCURRENT))
elapsed   = time.perf_counter() - t_start

latencies = [r["latency"] for r in results]

print(f"Processed {len(results)} requests in {elapsed:.2f}s  "
      f"(sequential would take ~{sum(latencies):.2f}s)")
print(f"Mean latency: {np.mean(latencies):.3f}s  "
      f"P95: {np.percentile(latencies, 95):.3f}s  "
      f"P99: {np.percentile(latencies, 99):.3f}s")

## 4.3 – Response Streaming for Low Latency

In [ ]:
# ---------------------------------------------------------------------------
# Server-Sent Events (SSE) streaming pattern
# ---------------------------------------------------------------------------
# Streaming lets the client display tokens as they are generated, reducing
# *perceived* latency from TTLT (time to last token) to TTFT (time to first
# token).
# ---------------------------------------------------------------------------

import sys

def stream_gemini_response(prompt: str, model_name: str = "gemini-1.5-pro"):
    """
    Stream tokens from Gemini and yield each chunk.

    In production:
        model    = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt, stream=True)
        for chunk in response:
            yield chunk.text

    Here we simulate streaming with delayed chunks.
    """
    simulated_tokens = [
        "Vertex ", "AI ", "is ", "Google ", "Cloud's ",
        "managed ", "machine ", "learning ", "platform ",
        "that ", "provides ", "everything ", "you ", "need ",
        "to ", "build ", "and ", "deploy ", "ML ", "models.",
    ]
    for token in simulated_tokens:
        yield token
        time.sleep(0.04)   # simulate inter-token delay


# Demonstrate streaming output
print("Streaming response: ", end="", flush=True)
full_text = ""
t_first   = None
t0        = time.perf_counter()

for chunk in stream_gemini_response("What is Vertex AI?"):
    if t_first is None:
        t_first = time.perf_counter() - t0
    print(chunk, end="", flush=True)
    full_text += chunk

t_total = time.perf_counter() - t0
print()   # newline
print(f"\nTTFT: {t_first:.3f}s  TTLT: {t_total:.3f}s  "
      f"Tokens: {len(full_text.split())}")

## 4.4 – Token Optimization and Prompt Engineering for Cost

In [ ]:
# ---------------------------------------------------------------------------
# Token counting with the Gemini SDK
# ---------------------------------------------------------------------------
# genai.GenerativeModel.count_tokens() returns a CountTokensResponse with
# a .total_tokens field.  Use this BEFORE sending the request to gate
# on cost or context-window limits.
# ---------------------------------------------------------------------------

def count_tokens_mock(text: str) -> int:
    """
    Approximate token count (Gemini uses SentencePiece tokenisation).
    Rule of thumb: ~1.3 tokens per English word.

    In production:
        model = genai.GenerativeModel('gemini-1.5-pro')
        result = model.count_tokens(text)
        return result.total_tokens
    """
    return int(len(text.split()) * 1.3)


# Compare verbose vs concise prompt
verbose_prompt = """
I would like you to please carefully read the following paragraph and then
provide me with a comprehensive and detailed summary that captures all of the
main points, key ideas, and important nuances contained within it, while
also making sure to highlight any significant conclusions or implications
that can be drawn from the content.

Paragraph: Transformer neural networks revolutionised NLP in 2017.
"""

concise_prompt = "Summarise in 2 sentences: Transformer neural networks revolutionised NLP in 2017."

verbose_tokens  = count_tokens_mock(verbose_prompt)
concise_tokens  = count_tokens_mock(concise_prompt)

print(f"Verbose prompt  : {verbose_tokens} tokens")
print(f"Concise prompt  : {concise_tokens} tokens")
print(f"Token reduction : {verbose_tokens - concise_tokens} "
      f"({(1 - concise_tokens/verbose_tokens)*100:.0f}%)")

# Project savings at scale
daily_requests = 10_000
pro_input_price_per_token = 3.50 / 1_000_000

verbose_daily_cost  = verbose_tokens  * daily_requests * pro_input_price_per_token
concise_daily_cost  = concise_tokens  * daily_requests * pro_input_price_per_token
monthly_savings     = (verbose_daily_cost - concise_daily_cost) * 30

print(f"\nAt {daily_requests:,} requests/day:")
print(f"  Verbose daily cost  : ${verbose_daily_cost:.4f}")
print(f"  Concise daily cost  : ${concise_daily_cost:.4f}")
print(f"  Estimated monthly savings: ${monthly_savings:.2f}")

## 4.5 – Load Testing Simulation and Throughput-Latency Trade-offs

In [ ]:
# ---------------------------------------------------------------------------
# Simulate throughput vs latency under different concurrency levels
# ---------------------------------------------------------------------------
# A real load test would use a tool like Locust or k6 against a live endpoint.
# Here we model the relationship analytically using Little's Law:
#   throughput = concurrency / mean_latency
# and a service time that grows with queue depth (M/M/1 queueing model).
# ---------------------------------------------------------------------------

# Service parameters
BASE_LATENCY_S    = 0.8   # latency at zero load (TTLT for a ~300-token response)
SERVICE_CAPACITY  = 20    # maximum sustainable RPS before the system saturates

concurrency_levels = np.arange(1, 25)

def model_latency(concurrency: int, base: float = BASE_LATENCY_S,
                  capacity: int = SERVICE_CAPACITY) -> float:
    """
    M/M/1 approximation: latency rises sharply as load approaches capacity.
    """
    rho = concurrency / capacity   # utilisation
    if rho >= 1.0:
        return float("inf")
    return base / (1.0 - rho)

latencies      = [model_latency(c) for c in concurrency_levels]
throughputs    = [c / l if l != float("inf") else 0
                  for c, l in zip(concurrency_levels, latencies)]

# Add realistic noise
np.random.seed(7)
noise_lat = np.random.normal(0, 0.05, len(concurrency_levels))
noise_thr = np.random.normal(0, 0.3,  len(concurrency_levels))
latencies_n   = [max(BASE_LATENCY_S, l + n) for l, n in zip(latencies,   noise_lat)]
throughputs_n = [max(0, t + n)              for t, n in zip(throughputs, noise_thr)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Concurrency vs Latency
axes[0].plot(concurrency_levels, latencies_n, marker="o", markersize=4)
axes[0].axvline(SERVICE_CAPACITY, color="red", linestyle="--",
                label=f"Capacity limit ({SERVICE_CAPACITY})")
axes[0].set_xlabel("Concurrency (in-flight requests)")
axes[0].set_ylabel("P50 Latency (s)")
axes[0].set_title("Latency vs Concurrency")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Concurrency vs Throughput
axes[1].plot(concurrency_levels, throughputs_n, marker="s", markersize=4, color="green")
axes[1].axvline(SERVICE_CAPACITY, color="red", linestyle="--",
                label=f"Capacity limit ({SERVICE_CAPACITY})")
axes[1].set_xlabel("Concurrency (in-flight requests)")
axes[1].set_ylabel("Throughput (RPS)")
axes[1].set_title("Throughput vs Concurrency")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Load Test Simulation – Gemini API Service (M/M/1 Model)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Identify the sweet spot (max throughput before latency doubles)
sweet_spot = next(
    (i for i, l in enumerate(latencies_n) if l > 2 * BASE_LATENCY_S),
    len(concurrency_levels) - 1,
)
print(f"Sweet spot: concurrency={concurrency_levels[sweet_spot]}  "
      f"throughput={throughputs_n[sweet_spot]:.1f} RPS  "
      f"latency={latencies_n[sweet_spot]:.2f}s")

In [ ]:
# ---------------------------------------------------------------------------
# Cost optimisation dashboard – compare model tiers across workload profiles
# ---------------------------------------------------------------------------

workloads = {
    "Chatbot (short context)": {"input": 300,  "output": 200,  "daily_calls": 50_000},
    "Document analysis (long)": {"input": 5000, "output": 500,  "daily_calls": 5_000},
    "Code generation": {"input": 800,  "output": 1200, "daily_calls": 20_000},
    "Batch summarisation": {"input": 2000, "output": 300,  "daily_calls": 100_000},
}

rows = []
for workload, specs in workloads.items():
    for model_name in ["gemini-1.5-pro", "gemini-1.5-flash"]:
        cost_per_call = estimate_cost(model_name, specs["input"], specs["output"])
        monthly_cost  = cost_per_call["total_cost_usd"] * specs["daily_calls"] * 30
        rows.append({
            "Workload":     workload,
            "Model":        model_name,
            "Daily calls":  specs["daily_calls"],
            "Cost/call ($)": cost_per_call["total_cost_usd"],
            "Monthly ($)":  round(monthly_cost, 2),
        })

cost_df = pd.DataFrame(rows)

# Pivot for comparison
pivot = cost_df.pivot_table(
    index="Workload", columns="Model",
    values="Monthly ($)", aggfunc="sum",
)
pivot["Savings (Flash vs Pro)"] = pivot["gemini-1.5-pro"] - pivot["gemini-1.5-flash"]
pivot["Savings %"] = (
    (pivot["gemini-1.5-pro"] - pivot["gemini-1.5-flash"]) /
    pivot["gemini-1.5-pro"] * 100
).round(1)

print("Monthly cost comparison (USD):")
print(pivot.to_string())

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(len(pivot.index))
width = 0.35

ax.bar(x - width/2, pivot["gemini-1.5-pro"],   width, label="gemini-1.5-pro")
ax.bar(x + width/2, pivot["gemini-1.5-flash"],  width, label="gemini-1.5-flash")

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=15, ha="right")
ax.set_ylabel("Monthly Cost (USD)")
ax.set_title("Monthly API Cost by Workload and Model Tier")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Summary

In this segment we covered the four pillars of production-grade Gemini deployments on GCP:

| Area | Key Tools | Key Takeaways |
|---|---|---|
| **MLOps** | Vertex AI Pipelines, Model Registry, Experiments | Automate the full model lifecycle; track every run |
| **Monitoring** | Model Monitoring, Explainable AI | Detect drift early; understand *why* the model predicts what it predicts |
| **Deployment** | Endpoints, FastAPI, Cloud Run | Use managed endpoints for custom models; Cloud Run for API wrappers |
| **Scaling** | asyncio, batching, caching, token optimisation | Concurrency + caching = lower cost and latency simultaneously |

**Next steps:**
- Explore [Vertex AI documentation](https://cloud.google.com/vertex-ai/docs)
- Check the [Gemini API pricing page](https://ai.google.dev/pricing)
- Try the hands-on exercises in the slide deck

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>